# Bài 2 — Bước 1 Prototype: Disentangled Concept–Residual Mapping trên Flora

Kiểm chứng tính khả thi của **Hướng 1 (Concept–Residual Mapping)** trên feature 256-D trích xuất sẵn (`STAR` / `SynSE`) và bộ 82 primitives sẵn có (dùng tạm từ `Flora/kg/` để test nhanh trước khi xây bộ Sub-Action Concept Bank riêng).

**Trước khi chạy**: commit và push thư mục `zeroshot/concept_map/` lên GitHub (`model_cr.py`, `trainer_cr.py`, `run_cr.py`). Repo private thì lưu token vào Colab Secrets với tên `GH_TOKEN`.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess, time, re
import numpy as np, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = '/content/drive/MyDrive/BaoHuy/flora_pretrain.zip'   # feature Flora đã trích sẵn trên Drive
OUT_DIR = f'{DRIVE_DIR}/concept_map_step1'                           # log + kết quả Bước 1
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], check=True, stdout=open(f'{OUT_DIR}/thesis_commit.txt', 'w'))

FLORA = f'{REPO}/zeroshot/Flora'
CMAP = f'{REPO}/zeroshot/concept_map'
need_flora = ['nesy_common.py', 'kg_semantics.py', 'kg/primitive_v2.json', 'kg/kg_rules_v2.json', 'semantics/ntu60_llm.txt', 'semantics/ntu120_llm.txt']
need_cmap = ['model_cr.py', 'trainer_cr.py', 'run_cr.py']
missing = [f'Flora/{p}' for p in need_flora if not os.path.exists(f'{FLORA}/{p}')]
missing += [f'concept_map/{p}' for p in need_cmap if not os.path.exists(f'{CMAP}/{p}')]
assert not missing, f'Repo chưa có {missing}: commit và push zeroshot/concept_map trước'
print('repo OK')

In [ ]:
!pip -q install transformers pyyaml pandas

# models/fm.py luôn import Shift-GCN, và Shift-GCN import CUDA op `shift_cuda`.
# Vì use_features=True nên Shift-GCN không bao giờ chạy, chỉ cần module rỗng để import qua.
stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
if not os.path.exists(stub):
    with open(stub, 'w') as f:
        f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!python -c "import sys; sys.path[:0]=['{FLORA}','{CMAP}']; from model_cr import ConceptResidualMappingNet; from trainer_cr import ConceptResidualTrainer; print('import OK')"

## Giải nén feature 256-D đã trích sẵn

Tận dụng đúng file `MyDrive/BaoHuy/flora_pretrain.zip` (`star_features`, `synse_features`), giải nén vào `/content/flora_features` rồi tạo symlink vào `Flora/skeleton_features/`. Sau mỗi lần restart runtime cần chạy lại cell này.

In [ ]:
assert os.path.exists(FEAT_ARCHIVE), f'Không thấy {FEAT_ARCHIVE} trên Drive'
!ls -lh {FEAT_ARCHIVE}

FEAT_NAMES = ('star_features', 'synse_features')

def find_feat(name):
    hits = [h for h in glob.glob(f'{EXTRACT}/**/{name}', recursive=True) if '__MACOSX' not in h]
    return hits[0] if hits else None

if not any(find_feat(n) for n in FEAT_NAMES):
    os.makedirs(EXTRACT, exist_ok=True)
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', FEAT_ARCHIVE, '-x', '__MACOSX/*', '*/._*', '._*', '-d', EXTRACT], check=True)
    print(f'giải nén xong trong {time.time() - t0:.0f}s')

assert any(find_feat(n) for n in FEAT_NAMES), f'Trong {FEAT_ARCHIVE} không có star_features hay synse_features'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)
for name in FEAT_NAMES:
    src_dir = find_feat(name)
    link = f'{FLORA}/skeleton_features/{name}'
    if src_dir is None:
        print(f'archive không có {name}')
        continue
    if os.path.islink(link):
        os.remove(link)
    os.symlink(src_dir, link)
    print(name, sorted(f for f in os.listdir(link) if not f.startswith('._')))

## Thử nghiệm Bước 1 (STAR features, 48/12 & 110/10, seed 1)

So sánh 5 cấu hình:
- `flora_base`: Flora gốc (`lambda_c=0, lambda_orth=0`)
- `cr_no_orth`: tách nhánh Concept (`D_c=256`) + Residual (`D_r=512`), CLIP-anchored head, chưa có trực giao
- `cr_orth`: thêm loss trực giao (`lambda_orth=0.1`) giữa $\mu_c$ và $\mu_r$
- `cr_linear`: thay CLIP-anchored head bằng Linear head (ablation)
- `cr_cbm_only`: Concept Bottleneck thuần (`D_c=768, D_r=0`, không có Residual)

In [ ]:
!python {CMAP}/run_cr.py --feat star --tasks 48 110 --seeds 1 --variants flora_base cr_no_orth cr_orth cr_linear cr_cbm_only --out-root {OUT_DIR}

## Xem chi tiết kết quả Concept Inference & Demo Top-5 concept (từ log)

In [ ]:
for split_task in ('ntu60_xsub_seen48_unseen12', 'ntu120_xsub_seen110_unseen10'):
    print(f'=== {split_task} (cr_orth) ===')
    log_path = f'{OUT_DIR}/star/cr_orth/{split_task}_ZSL_seed1/run.log'
    if os.path.exists(log_path):
        for line in open(log_path, encoding='utf-8'):
            if '[CR-EVAL]' in line or '[CR-DEMO]' in line or 'Best ZSL' in line:
                print(line.strip())
    else:
        print('Chưa có log:', log_path)

## (Tùy chọn) Kiểm chứng thêm trên SynSE features hoặc quét trọng số mạnh hơn (`cr_orth_s2`)

In [ ]:
# !python {CMAP}/run_cr.py --feat synse --tasks 48 110 --seeds 1 --variants flora_base cr_orth --out-root {OUT_DIR}
# !python {CMAP}/run_cr.py --feat star --tasks 48 110 --seeds 1 --variants cr_orth_s2 --out-root {OUT_DIR}